In [0]:
##07/11/2026
Este notebook tiene la responsabilidad de realizar la ingesta de los datos de facturacion ubicadod en /raw e insertarlos en la capa bronze

In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_pysmartdata")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "pysmartdata2026")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/1.Facturacion_Acueducto_202607-202609.csv"

In [0]:
df_facturacion = spark.read.option('header', True)\
                        .option('sep', ';')\
                        .option('inferSchema', True)\
                        .csv(ruta)

In [0]:
facturacion_schema = StructType(fields=[StructField("periodo", IntegerType(), True),
                                     StructField("suscriptor", IntegerType(), True),
                                     StructField("nombre_cliente", StringType(), True),
                                     StructField("direccion", StringType(), True),
                                     StructField("estrato", IntegerType(), True),
                                
                                     StructField("codigo_concepto", IntegerType(), True),
                                     StructField("concepto", StringType(), True),
                                     StructField("consumo_m3", IntegerType(), True),
                                     # Los valores monetarios vienen con coma decimal (ej. 7785,96): se leen como texto y se convierten a double en la celda de selección
                                     StructField("valor_cargo_fijo", StringType(), True),
                                     StructField("valor_consumo", StringType(), True),
                                     StructField("valor_total", StringType(), True),
                                     StructField("ingestion_date", TimestampType(), True)
])

In [0]:
df_facturacion_final = spark.read\
.option('header', True)\
.option('sep', ';')\
.schema(facturacion_schema)\
.csv(ruta)

In [0]:
facturacion_selected_df = df_facturacion_final.select(col("periodo"), 
                                                col("suscriptor"), 
                                                col("nombre_cliente"), 
                                                col("direccion"), 
                                                col("estrato"), 
                                                col("codigo_concepto"), 
                                                col("concepto"), 
                                                col("consumo_m3"), 
                                                regexp_replace(col("valor_cargo_fijo"), ",", ".").cast("double").alias("valor_cargo_fijo"), 
                                                regexp_replace(col("valor_consumo"), ",", ".").cast("double").alias("valor_consumo"), 
                                                regexp_replace(col("valor_total"), ",", ".").cast("double").alias("valor_total"),
                                                col("ingestion_date"))

In [0]:
facturacion_renamed_df = facturacion_selected_df.withColumnRenamed("periodo", "periodo") \
                                            .withColumnRenamed("suscriptor", "suscriptor") \
                                            .withColumnRenamed("nombre_cliente", "nombre_cliente") \
                                            .withColumnRenamed("direccion", "direccion") \
                                            .withColumnRenamed("estrato", "estrato") \
                                            .withColumnRenamed("codigo_concepto", "codigo_concepto") \
                                            .withColumnRenamed("concepto", "concepto") \
                                            .withColumnRenamed("consumo_m3", "consumo_m3") \
                                            .withColumnRenamed("valor_cargo_fijo", "valor_cargo_fijo") \
                                            .withColumnRenamed("valor_consumo", "valor_consumo") \
                                            .withColumnRenamed("ingestion_date", "ingestion_date")
 

In [0]:
facturacion_final_df = facturacion_renamed_df.withColumn("ingestion_date", current_timestamp())

In [0]:
facturacion_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.facturacion")

In [0]:
tabla = f"{catalogo}.{esquema}.facturacion"
df_tabla = spark.table(tabla)

# 1. Comparar conteo de filas: origen (CSV) vs tabla destino
filas_origen = facturacion_final_df.count()
filas_tabla = df_tabla.count()
print(f"Filas origen: {filas_origen} | Filas en tabla: {filas_tabla} | Coinciden: {filas_origen == filas_tabla}")

# 2. Resumen por periodo, nulos en llaves y fecha de ingesta
display(
    df_tabla.groupBy("periodo").agg(
        count("*").alias("filas"),
        countDistinct("suscriptor").alias("suscriptores"),
        sum(when(col("suscriptor").isNull(), 1).otherwise(0)).alias("suscriptor_nulo"),
        round(sum("valor_consumo"), 2).alias("total_valor_consumo"),
        max("ingestion_date").alias("ultima_ingesta")
    ).orderBy("periodo")
)

# 3. Muestra de registros
display(df_tabla.limit(10))